# Análise Preliminar e Exploratória de Dados
## Dataset: Cancro da Mama Wisconsin (Breast Cancer)

**Disciplina:** Análise Inteligente de Dados  
**Curso:** Engenharia Biomédica - ISEC  

---

## Objetivos da Aula

1. Realizar **análise preliminar** de dados de imagiologia médica
2. Executar **análise exploratória** de características de tumores
3. Compreender a estrutura de features extraídas de imagens médicas
4. Identificar padrões que distinguem tumores benignos de malignos

---

## Contexto do Dataset

Este dataset contém características computadas de imagens digitalizadas de aspirados por agulha fina (FNA) de massas mamárias. As características descrevem propriedades dos núcleos celulares presentes nas imagens.

**Estrutura das Variáveis:**

Para cada núcleo celular, 10 características são medidas:
1. `radius`: Raio (média das distâncias do centro aos pontos do perímetro)
2. `texture`: Textura (desvio padrão dos valores em escala de cinza)
3. `perimeter`: Perímetro
4. `area`: Área
5. `smoothness`: Suavidade (variação local nos comprimentos do raio)
6. `compactness`: Compacidade (perímetro² / área - 1.0)
7. `concavity`: Concavidade (severidade das porções côncavas do contorno)
8. `concave_points`: Pontos côncavos (número de porções côncavas do contorno)
9. `symmetry`: Simetria
10. `fractal_dimension`: Dimensão fractal ("aproximação da linha costeira" - 1)

**Para cada característica, são calculadas 3 medidas:**
- Média (`_mean`)
- Erro padrão (`_se`)
- "Pior" valor ou maior (`_worst`)

Resultando em 30 features numéricas + 1 variável alvo (diagnóstico)

**Variável Alvo:**
- `diagnosis`: M = Maligno, B = Benigno


## 1. Importação de Bibliotecas

In [ ]:
# Bibliotecas essenciais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Configurações de visualização
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Configuração para exibir todas as colunas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 100)

print("✓ Bibliotecas importadas com sucesso!")

## 2. Carregamento dos Dados

In [ ]:
# Carregar dataset do sklearn
from sklearn.datasets import load_breast_cancer

# Carregar dados
data = load_breast_cancer()

# Criar DataFrame
df = pd.DataFrame(data.data, columns=data.feature_names)
df['diagnosis'] = data.target

# Converter target para labels descritivos
df['diagnosis_label'] = df['diagnosis'].map({0: 'M', 1: 'B'})  # 0=Maligno, 1=Benigno

print("✓ Dados carregados com sucesso!")
print(f"Dimensões do dataset: {df.shape[0]} linhas × {df.shape[1]} colunas")
print(f"\nDescrição do dataset:")
print(data.DESCR[:500] + "...")

---
# PARTE I - ANÁLISE PRELIMINAR
---

## 3. Primeiras Observações

In [ ]:
# Visualizar primeiras linhas
print("=== PRIMEIRAS 5 LINHAS ===")
df.head()

In [ ]:
# Visualizar últimas linhas
print("=== ÚLTIMAS 5 LINHAS ===")
df.tail()

In [ ]:
# Amostra aleatória
print("=== AMOSTRA ALEATÓRIA (5 registos) ===")
df.sample(5, random_state=42)

### 🤔 Questão para Reflexão

Observa que temos muitas features (30). Elas estão agrupadas em categorias (mean, error, worst). Porquê esta estrutura?

## 4. Informações Gerais sobre o Dataset

In [ ]:
# Informação geral
print("=== INFORMAÇÕES GERAIS ===")
df.info()

In [ ]:
# Organizar features por tipo
mean_features = [col for col in df.columns if 'mean' in col]
se_features = [col for col in df.columns if 'error' in col]
worst_features = [col for col in df.columns if 'worst' in col]

print("=== ORGANIZAÇÃO DAS FEATURES ===")
print(f"\nFeatures MEAN ({len(mean_features)}): Média das medições")
print(mean_features)
print(f"\nFeatures SE ({len(se_features)}): Erro padrão")
print(se_features)
print(f"\nFeatures WORST ({len(worst_features)}): Valores máximos/piores")
print(worst_features)

## 5. Análise de Valores em Falta

In [ ]:
# Contagem de valores em falta
print("=== VALORES EM FALTA ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Valores em Falta': missing,
    'Percentagem (%)': missing_pct
})

if missing.sum() == 0:
    print("✓ Não existem valores em falta no dataset!")
else:
    print(missing_df[missing_df['Valores em Falta'] > 0])

## 6. Estatísticas Descritivas

In [ ]:
# Estatísticas descritivas gerais
print("=== ESTATÍSTICAS DESCRITIVAS - TODAS AS FEATURES ===")
df.describe().round(2)

In [ ]:
# Estatísticas por grupo de features
print("=== ESTATÍSTICAS - FEATURES MEAN ===")
print(df[mean_features].describe().round(2))

In [ ]:
# Análise de variabilidade
print("=== ANÁLISE DE VARIABILIDADE ===")
variabilidade = pd.DataFrame({
    'Desvio Padrão': df[mean_features].std(),
    'Coef. Variação (%)': (df[mean_features].std() / df[mean_features].mean() * 100)
}).sort_values('Coef. Variação (%)', ascending=False).round(2)

print(variabilidade)

### ✏️ Exercício 1

**Observando as estatísticas descritivas:**
1. Qual feature tem maior variabilidade?
2. As escalas das variáveis são muito diferentes?
3. O que isso implica para análise e modelação futura?

## 7. Análise de Duplicados

In [ ]:
# Verificar duplicados
print("=== ANÁLISE DE DUPLICADOS ===")
duplicados = df.duplicated().sum()
print(f"Número de linhas duplicadas: {duplicados}")
print(f"Percentagem de duplicados: {(duplicados/len(df)*100):.2f}%")

---
# PARTE II - ANÁLISE EXPLORATÓRIA
---

## 8. Análise da Variável Alvo (Diagnosis)

In [ ]:
# Distribuição do diagnóstico
print("=== DISTRIBUIÇÃO DO DIAGNÓSTICO ===")
diag_counts = df['diagnosis_label'].value_counts().sort_index()
diag_pct = df['diagnosis_label'].value_counts(normalize=True).sort_index() * 100

diag_df = pd.DataFrame({
    'Contagem': diag_counts,
    'Percentagem (%)': diag_pct.round(2)
})
diag_df.index = ['Benigno (B)', 'Maligno (M)']
print(diag_df)

In [ ]:
# Visualização da distribuição
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
colors = ['#2ecc71', '#e74c3c']
diag_counts.plot(kind='bar', ax=ax1, color=colors)
ax1.set_title('Distribuição de Diagnósticos', fontsize=14, fontweight='bold')
ax1.set_xlabel('Diagnóstico', fontsize=12)
ax1.set_ylabel('Número de Casos', fontsize=12)
ax1.set_xticklabels(['Benigno', 'Maligno'], rotation=0)
ax1.grid(axis='y', alpha=0.3)

# Gráfico de pizza
ax2.pie(diag_counts, labels=['Benigno', 'Maligno'], autopct='%1.1f%%',
        colors=colors, startangle=90, textprops={'fontsize': 12})
ax2.set_title('Proporção de Diagnósticos', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 🤔 Questão para Reflexão

O dataset está balanceado? Em contexto clínico, o que aconteceria se tivéssemos um dataset muito desbalanceado?

## 9. Distribuições das Features MEAN

In [ ]:
# Histogramas das features MEAN
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.ravel()

for idx, col in enumerate(mean_features):
    axes[idx].hist(df[col], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{col}', fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('Valor', fontsize=9)
    axes[idx].set_ylabel('Frequência', fontsize=9)
    axes[idx].grid(axis='y', alpha=0.3)

# Remover subplots extra
for i in range(len(mean_features), len(axes)):
    fig.delaxes(axes[i])

plt.suptitle('Distribuições das Features MEAN', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Distribuições com KDE
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.ravel()

for idx, col in enumerate(mean_features):
    sns.histplot(df[col], kde=True, ax=axes[idx], color='teal', bins=30)
    axes[idx].set_title(f'{col}', fontsize=10, fontweight='bold')

# Remover subplots extra
for i in range(len(mean_features), len(axes)):
    fig.delaxes(axes[i])

plt.suptitle('Distribuições com KDE - Features MEAN', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 10. Análise de Outliers

In [ ]:
# Boxplots das features MEAN
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.ravel()

for idx, col in enumerate(mean_features):
    axes[idx].boxplot(df[col], vert=True, patch_artist=True,
                     boxprops=dict(facecolor='lightblue', color='navy'),
                     medianprops=dict(color='red', linewidth=2),
                     whiskerprops=dict(color='navy'),
                     capprops=dict(color='navy'))
    axes[idx].set_title(f'{col}', fontsize=10, fontweight='bold')
    axes[idx].set_ylabel('Valor', fontsize=9)
    axes[idx].grid(axis='y', alpha=0.3)

# Remover subplots extra
for i in range(len(mean_features), len(axes)):
    fig.delaxes(axes[i])

plt.suptitle('Boxplots - Features MEAN', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Contagem de outliers
print("=== DETECÇÃO DE OUTLIERS (Método IQR) - Features MEAN ===")
print("\nNúmero de outliers por feature:")

for col in mean_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
    print(f"{col:30s}: {len(outliers):3d} outliers ({len(outliers)/len(df)*100:.1f}%)")

## 11. Matriz de Correlação

In [ ]:
# Matriz de correlação - apenas features MEAN
correlation_matrix_mean = df[mean_features + ['diagnosis']].corr()

print("=== MATRIZ DE CORRELAÇÃO - Features MEAN ===")
print(correlation_matrix_mean.round(2))

In [ ]:
# Heatmap da matriz de correlação
plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix_mean, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            annot_kws={'size': 8})
plt.title('Matriz de Correlação - Features MEAN + Diagnosis', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlações com diagnosis
print("=== CORRELAÇÕES COM DIAGNOSIS (Maligno vs Benigno) ===")
diag_corr = correlation_matrix_mean['diagnosis'].drop('diagnosis').sort_values(ascending=False)
print(diag_corr.round(3))

# Visualização
plt.figure(figsize=(10, 8))
diag_corr.plot(kind='barh', color=['red' if x < 0 else 'green' for x in diag_corr])
plt.title('Correlação das Features MEAN com Diagnosis', fontsize=14, fontweight='bold')
plt.xlabel('Coeficiente de Correlação', fontsize=12)
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### ✏️ Exercício 2

**Analise as correlações:**
1. Quais features têm correlação negativa forte com malignidade?
2. Quais features estão altamente correlacionadas entre si?
3. O que significa ter features muito correlacionadas (multicolinearidade)?

## 12. Comparação entre Tumores Benignos e Malignos

In [ ]:
# Boxplots comparativos - Features MEAN
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.ravel()

for idx, col in enumerate(mean_features):
    df.boxplot(column=col, by='diagnosis_label', ax=axes[idx], patch_artist=True)
    axes[idx].set_title(f'{col}', fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('Diagnóstico (B=Benigno, M=Maligno)', fontsize=8)
    axes[idx].set_ylabel('Valor', fontsize=9)

# Remover subplots extra
for i in range(len(mean_features), len(axes)):
    fig.delaxes(axes[i])

plt.suptitle('Comparação Benigno vs Maligno - Features MEAN', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Violin plots para features selecionadas
top_features = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 
                'mean concavity', 'mean concave points']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, col in enumerate(top_features):
    sns.violinplot(data=df, x='diagnosis_label', y=col, ax=axes[idx], 
                   palette={'B': '#2ecc71', 'M': '#e74c3c'})
    axes[idx].set_title(f'{col}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Diagnóstico', fontsize=10)
    axes[idx].set_ylabel('Valor', fontsize=10)

plt.suptitle('Violin Plots - Features Principais', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 13. Estatísticas por Grupo (Diagnosis)

In [ ]:
# Comparação de médias - Features MEAN
print("=== COMPARAÇÃO DE MÉDIAS POR DIAGNÓSTICO ===")
comparacao = pd.DataFrame({
    'Benigno (média)': df[df['diagnosis_label'] == 'B'][mean_features].mean(),
    'Maligno (média)': df[df['diagnosis_label'] == 'M'][mean_features].mean(),
    'Diferença': df[df['diagnosis_label'] == 'M'][mean_features].mean() - 
                 df[df['diagnosis_label'] == 'B'][mean_features].mean(),
    'Diferença (%)': ((df[df['diagnosis_label'] == 'M'][mean_features].mean() - 
                       df[df['diagnosis_label'] == 'B'][mean_features].mean()) / 
                      df[df['diagnosis_label'] == 'B'][mean_features].mean() * 100)
}).round(2)

print(comparacao)

In [ ]:
# Visualização das diferenças percentuais
plt.figure(figsize=(10, 8))
comparacao['Diferença (%)'].sort_values().plot(kind='barh', 
                                                color=['green' if x < 0 else 'red' for x in comparacao['Diferença (%)'].sort_values()])
plt.title('Diferença Percentual nas Médias (Maligno vs Benigno)', fontsize=14, fontweight='bold')
plt.xlabel('Diferença Percentual (%)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### ✏️ Exercício 3

**Observando as diferenças:**
1. Quais features mostram maior diferença entre benignos e malignos?
2. Há features onde a diferença é pequena?
3. Isso faz sentido biologicamente?

## 14. Análise de Relações entre Features

In [ ]:
# Scatterplot: Radius vs Area
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'B': '#2ecc71', 'M': '#e74c3c'}
for diag in ['B', 'M']:
    subset = df[df['diagnosis_label'] == diag]
    label = 'Benigno' if diag == 'B' else 'Maligno'
    ax.scatter(subset['mean radius'], subset['mean area'], c=colors[diag], 
               label=label, alpha=0.6, s=50, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Mean Radius', fontsize=12)
ax.set_ylabel('Mean Area', fontsize=12)
ax.set_title('Relação entre Radius e Area', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatterplot: Concavity vs Concave Points
fig, ax = plt.subplots(figsize=(10, 6))

for diag in ['B', 'M']:
    subset = df[df['diagnosis_label'] == diag]
    label = 'Benigno' if diag == 'B' else 'Maligno'
    ax.scatter(subset['mean concavity'], subset['mean concave points'], c=colors[diag], 
               label=label, alpha=0.6, s=50, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Mean Concavity', fontsize=12)
ax.set_ylabel('Mean Concave Points', fontsize=12)
ax.set_title('Relação entre Concavity e Concave Points', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 15. Pairplot de Features Principais

In [ ]:
# Pairplot das 6 features mais importantes
principais = ['mean radius', 'mean texture', 'mean perimeter', 
              'mean concavity', 'mean concave points', 'diagnosis_label']

sns.pairplot(df[principais], hue='diagnosis_label', 
             palette={'B': '#2ecc71', 'M': '#e74c3c'},
             plot_kws={'alpha': 0.6}, diag_kind='kde')
plt.suptitle('Pairplot - Features Principais', y=1.01, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 16. Análise Comparativa: MEAN vs WORST

In [ ]:
# Comparar radius mean vs worst
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Mean radius
for diag in ['B', 'M']:
    subset = df[df['diagnosis_label'] == diag]
    label = 'Benigno' if diag == 'B' else 'Maligno'
    ax1.scatter(subset.index, subset['mean radius'], c=colors[diag], 
               label=label, alpha=0.5, s=30)

ax1.set_xlabel('Índice da Amostra', fontsize=11)
ax1.set_ylabel('Mean Radius', fontsize=11)
ax1.set_title('Mean Radius por Diagnóstico', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Worst radius
for diag in ['B', 'M']:
    subset = df[df['diagnosis_label'] == diag]
    label = 'Benigno' if diag == 'B' else 'Maligno'
    ax2.scatter(subset.index, subset['worst radius'], c=colors[diag], 
               label=label, alpha=0.5, s=30)

ax2.set_xlabel('Índice da Amostra', fontsize=11)
ax2.set_ylabel('Worst Radius', fontsize=11)
ax2.set_title('Worst Radius por Diagnóstico', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Comparar poder discriminativo: MEAN vs WORST
correlation_worst = df[worst_features + ['diagnosis']].corr()['diagnosis'].drop('diagnosis').abs()
correlation_mean_abs = diag_corr.abs()

# Criar dicionário de comparação
feature_types = ['radius', 'texture', 'perimeter', 'area', 'smoothness', 
                 'compactness', 'concavity', 'concave points', 'symmetry', 'fractal dimension']

comparacao_mean_worst = pd.DataFrame({
    'MEAN (|corr|)': [correlation_mean_abs[f'mean {ft}'] for ft in feature_types],
    'WORST (|corr|)': [correlation_worst[f'worst {ft}'] for ft in feature_types]
}, index=feature_types)

print("=== COMPARAÇÃO: Correlação Absoluta com Diagnosis ===")
print(comparacao_mean_worst.round(3))

In [ ]:
# Visualização da comparação
fig, ax = plt.subplots(figsize=(12, 8))

x = np.arange(len(feature_types))
width = 0.35

bars1 = ax.bar(x - width/2, comparacao_mean_worst['MEAN (|corr|)'], width, 
               label='MEAN', color='steelblue')
bars2 = ax.bar(x + width/2, comparacao_mean_worst['WORST (|corr|)'], width, 
               label='WORST', color='coral')

ax.set_xlabel('Feature Type', fontsize=12)
ax.set_ylabel('Correlação Absoluta com Diagnosis', fontsize=12)
ax.set_title('Poder Discriminativo: MEAN vs WORST Features', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(feature_types, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 🤔 Questão para Reflexão

As features WORST têm geralmente maior correlação com malignidade. Porque será que os valores "piores" são mais informativos?

---
## 17. Síntese e Conclusões
---

### 📊 Principais Descobertas

**Análise Preliminar:**
- Dataset com 569 amostras e 30 features numéricas
- **Excelente qualidade:** Sem valores em falta, sem duplicados
- Dataset razoavelmente balanceado: 63% benignos, 37% malignos
- Features organizadas em 3 grupos: MEAN, SE (error), WORST

**Características das Features:**
- **Escalas muito diferentes:** área (143-2501) vs simetria (0.1-0.3)
- **Alta variabilidade:** especialmente em área, perímetro e textura
- **Multicolinearidade:** muitas features correlacionadas entre si (ex: radius, perimeter, area)

**Análise Exploratória:**
- **Features mais correlacionadas com malignidade (negativas = pior):**
  - Concave points mean: -0.78
  - Perimeter mean: -0.74
  - Concavity mean: -0.70
  - Radius mean: -0.73
  - Area mean: -0.71

**Diferenças Benigno vs Maligno:**
- Tumores malignos são consistentemente **maiores**
- Apresentam **maior concavidade** e mais **pontos côncavos**
- **Perímetro** e **área** muito superiores (+86% e +113%)
- **Textura** mais irregular (+34%)

**MEAN vs WORST:**
- Features WORST têm **maior poder discriminativo**
- Capturam características extremas/anormais do tumor
- Concave points worst tem correlação de -0.79 (vs -0.78 para mean)

**Outliers:**
- Presentes em várias features, especialmente área e perímetro
- Podem representar casos reais de tumores muito grandes
- Importante analisar caso a caso

**Próximos Passos Sugeridos:**
1. Normalização/padronização das features (escalas muito diferentes)
2. Análise de componentes principais (PCA) - muita multicolinearidade
3. Feature selection - 30 features pode ser redundante
4. Análise mais detalhada das features SE (erro padrão)
5. Preparação para modelação

### ✏️ Exercício Final

**Reflexão Crítica:**

1. **Sobre as Features:**
   - Porque é que temos 30 features? É necessário usar todas?
   - Qual a vantagem de ter MEAN, SE e WORST para cada medida?
   - Como a multicolinearidade pode afetar uma análise futura?

2. **Interpretação Clínica:**
   - Faz sentido que tumores malignos sejam maiores e mais irregulares?
   - Que outras características imagiológicas poderiam ser relevantes?
   - Como estas features se relacionam com o que vês numa mamografia?

3. **Qualidade e Preparação:**
   - Este dataset está "demasiado limpo"? É realista?
   - Que pré-processamento seria essencial antes de modelar?
   - Como lidarias com as diferentes escalas das features?

4. **Aplicação Prática:**
   - Se fosses criar um modelo preditivo, que features escolherias?
   - Como validarias o modelo num contexto clínico real?
   - Que precauções éticas devias ter ao usar este tipo de modelo?

---
## 📚 Referências e Recursos

**Dataset:**
- UCI Machine Learning Repository - Breast Cancer Wisconsin (Diagnostic) Dataset
- Dr. William H. Wolberg, General Surgery Dept., University of Wisconsin
- Criado a partir de imagens digitalizadas de FNA (Fine Needle Aspiration)

**Publicações Relacionadas:**
- W.N. Street, W.H. Wolberg and O.L. Mangasarian (1993)
  "Nuclear feature extraction for breast tumor diagnosis"
  IS&T/SPIE 1993 International Symposium on Electronic Imaging

**Documentação:**
- Pandas: https://pandas.pydata.org/
- Matplotlib: https://matplotlib.org/
- Seaborn: https://seaborn.pydata.org/
- Scikit-learn: https://scikit-learn.org/

---
**Engenharia Biomédica - IPC**  
*Análise Inteligente de Dados*